> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t06_bronze_to_silver_after_drift` · notebook `02_bronze_to_silver` · started 2026-10-11 13:03 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `mode=incremental`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 02 · Bronze → Silver
Casts epoch integers to timestamps, decodes titles, removes PII (drops names/avatars, hashes user ids with a
salted SHA-256), derives body features, applies data-quality rules and upserts every Silver table with
**`MERGE INTO`**. Re-running the same batches changes nothing (idempotent).

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: Bronze batches not yet in Silver | `backfill`: re-process the selected batches |
| `batch_ids` | empty | comma-separated batch ids |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `pii_salt` | empty (read from secret `so_lakehouse/pii_salt`) | same |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("pii_salt", "")

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.silver import run_bronze_to_silver

cfg = build_config(need_salt=True)
audit = run_bronze_to_silver(
    spark, cfg,
    mode=widget("mode"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    fail_on_error=True,
)

Bronze-to-Silver: processing 1 batch(es): incr_20261008T000000Z
[SUCCESS] Bronze-to-Silver | questions                          | read=2 ins=2 upd=0 del=0 quar=0 | 23.7s | 2 unique questions after de-duplication
[SUCCESS] Bronze-to-Silver | question_tags                      | read=8 ins=8 upd=0 del=0 quar=0 | 12.2s
[SUCCESS] Bronze-to-Silver | answers                            | read=0 ins=0 upd=0 del=0 quar=0 | 6.4s | 0 unique answers after de-duplication
[SUCCESS] Bronze-to-Silver | users                              | read=2 ins=0 upd=0 del=0 quar=0 | 7.5s
[SUCCESS] Bronze-to-Silver | question_snapshots                 | read=0 ins=0 upd=0 del=0 quar=0 | 4.3s
[SUCCESS] Bronze-to-Silver | question_metrics (from snapshots)  | read=0 ins=0 upd=0 del=0 quar=0 | 4.3s
[SUCCESS] Bronze-to-Silver | deleted_questions -> questions     | read=0 ins=0 upd=0 del=0 quar=0 | 4.9s | soft delete (is_deleted = true)
[SUCCESS] Bronze-to-Silver | deleted_questions -> answers       | read=0 ins=0 upd=

In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
bedef9cf-5583-4317-8a0d-d6e7c8f50303,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,questions,,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.questions,2026-10-11T13:03:31.235Z,2026-10-11T13:03:54.894Z,23.6591,SUCCESS,2,2,0,0,0,2 unique questions after de-duplication,2026-10-11T13:03:54.894Z
e671c2b0-b0c7-4b4f-8971-eebddb62fda4,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,question_tags,,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.question_tags,2026-10-11T13:03:56.584Z,2026-10-11T13:04:08.834Z,12.249839,SUCCESS,8,8,0,0,0,,2026-10-11T13:04:08.834Z
cff0cef9-837e-43d9-859e-2f8594b42c8a,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,answers,,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.answers,2026-10-11T13:04:10.240Z,2026-10-11T13:04:16.678Z,6.437767,SUCCESS,0,0,0,0,0,0 unique answers after de-duplication,2026-10-11T13:04:16.678Z
feda4aa3-7863-4b37-b47b-463d7d7d6a12,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,users,,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.users,2026-10-11T13:04:18.185Z,2026-10-11T13:04:25.720Z,7.535405,SUCCESS,2,0,0,0,0,,2026-10-11T13:04:25.720Z
c3b46a13-3c40-4292-a010-8d436dfb4c9a,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,question_snapshots,,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.question_snapshots,2026-10-11T13:04:27.166Z,2026-10-11T13:04:31.432Z,4.266335,SUCCESS,0,0,0,0,0,,2026-10-11T13:04:31.432Z
a85f2e58-507c-4087-9568-d7486745039e,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,question_metrics (from snapshots),,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.questions,2026-10-11T13:04:32.934Z,2026-10-11T13:04:37.247Z,4.313211,SUCCESS,0,0,0,0,0,,2026-10-11T13:04:37.247Z
9a11087c-0eb1-4be4-941b-b6977d0bed56,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,deleted_questions -> questions,,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.questions,2026-10-11T13:04:38.570Z,2026-10-11T13:04:43.509Z,4.9386,SUCCESS,0,0,0,0,0,soft delete (is_deleted = true),2026-10-11T13:04:43.509Z
d59a5b61-d8fc-488f-a1d6-8d5587921446,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,deleted_questions -> answers,,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.answers,2026-10-11T13:04:44.928Z,2026-10-11T13:04:50.029Z,5.100873,SUCCESS,0,0,0,0,0,soft delete (is_deleted = true),2026-10-11T13:04:50.029Z
8c925351-174f-46b2-8ea2-11178fe3b89d,bronze_to_silver-20261011T130329Z-5bb8e1,bronze_to_silver,Bronze-to-Silver,incremental,incremental,post_text_masked,,unprocessed batches -> [incr_20261008T000000Z],workspace.so_silver.post_text_masked,2026-10-11T13:04:52.273Z,2026-10-11T13:05:03.047Z,10.774204,SUCCESS,2,2,0,0,0,,2026-10-11T13:05:03.047Z
